# Blue Book for Bulldozers — Time-Aware Model Development

This is the canonical modelling companion. It uses tested helpers in `src/bulldozer_pipeline.py` so date handling, temporal splitting, RMSLE, missing values, and novel categories follow one reproducible contract.

It is unexecuted because the authorised competition CSVs are absent from Git.


## Evaluation contract

```text
historical records before cutoff → fit preprocessing + forest
later records at/after cutoff   → validate once with RMSLE
```

The competition is temporal. A random split is not a valid substitute because it lets later market context enter training.


In [ ]:
import pandas as pd
from src.bulldozer_pipeline import add_sale_date_features, build_regression_pipeline, rmsle, temporal_split

DATA_PATH = "data/bluebook-for-bulldozers/TrainAndValid.csv"
CUTOFF = "2012-01-01"  # Historical rows before 2012 train; 2012 rows validate.
raw = pd.read_csv(DATA_PATH, low_memory=False, parse_dates=["saledate"])
raw = raw.sort_values("saledate")


## 1. Preserve chronology before preprocessing

`temporal_split` is called on raw dates. Date features are created only after the partition is chosen; this keeps the temporal decision visible and prevents accidental future-row fitting.


In [ ]:
train_raw, valid_raw = temporal_split(raw, cutoff=CUTOFF)
X_train = add_sale_date_features(train_raw.drop(columns="SalePrice"))
y_train = train_raw["SalePrice"]
X_valid = add_sale_date_features(valid_raw.drop(columns="SalePrice"))
y_valid = valid_raw["SalePrice"]
print(f"Train: {len(X_train):,} rows | Validation: {len(X_valid):,} rows")


## 2. Fit a category-safe baseline

The pipeline learns medians and compact category codes from training rows only. At validation or test time, missing numeric values receive training medians, missing categories receive training modes, and previously unseen categories map to a fixed sentinel rather than forcing manual column alignment.


In [ ]:
pipeline = build_regression_pipeline(random_state=42, n_estimators=120)
pipeline.fit(X_train, y_train)
valid_predictions = pipeline.predict(X_valid).clip(min=0)
validation_rmsle = rmsle(y_valid, valid_predictions)
print(f"Validation RMSLE: {validation_rmsle:.4f}")


## 3. Interpret the score honestly

RMSLE measures multiplicative error. Report it with date coverage, validation size, and residual checks—not as a general equipment valuation guarantee. Before a final test submission, preprocess `Test.csv` with the same helpers and use the fitted pipeline directly; do not refit it on validation/test data.
